In [2]:
import sys
from pathlib import Path
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [3]:
from config import (
    BEST_HUMAN_CUES_PATH,
    SEED_EXAMPLES_PATH,
    BEST_PROMPT_BY_CATEGORY_PATH,
    FINAL_GENERATED_CUES_BY_CATEGORY_PATH,
)
from prompt_utils import sample_demo_examples, format_demo_block
from llm_utils import generate_therapy_cue
from scoring_utils import compute_proxy_score

Using local model path: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\models\Qwen2.5-1.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


In [4]:
best_human_df = pd.read_csv(BEST_HUMAN_CUES_PATH)
seed_df = pd.read_csv(SEED_EXAMPLES_PATH)

print("BEST_PROMPT_BY_CATEGORY_PATH:", BEST_PROMPT_BY_CATEGORY_PATH)
print("File exists:", BEST_PROMPT_BY_CATEGORY_PATH.exists())

best_prompt_by_category_df = pd.read_csv(BEST_PROMPT_BY_CATEGORY_PATH)

print("BEST PROMPT DF SHAPE:", best_prompt_by_category_df.shape)
display(best_prompt_by_category_df.head(20))

best_prompt_map = dict(
    zip(best_prompt_by_category_df["subcategory"], best_prompt_by_category_df["prompt_text"])
)

print("Categories with best prompts:", len(best_prompt_map))
print(sorted(best_prompt_map.keys()))

BEST_PROMPT_BY_CATEGORY_PATH: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\final\best_prompt_by_category.csv
File exists: True
BEST PROMPT DF SHAPE: (21, 10)


,subcategory,prompt_text,mean_proxy_score,ape_round,parent_prompt_id,prompt_id,prompt_family,prompt_variant,variant_bonus,selection_score
0,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,0.595717,0,NaN,交通工具_p02,category_meta,manual_prof_override,0.05,0.645717
1,休閒娛樂,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.686101,0,NaN,休閒娛樂_p01,category_meta,v1_meta_base,0.03,0.716101
2,動作,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,0.548430,0,NaN,動作_p03,category_meta,v3_life_context,0.04,0.588430
3,動物,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動物詞彙，並生成...,0.697142,0,NaN,動物_p02,category_meta,v2_discriminative,0.05,0.747142
4,喝,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.989949,0,NaN,喝_p02,category_meta,v2_discriminative,0.05,1.039949
5,娛樂場所,描述地點的常見景象或功能，避免太抽象的描述。,0.353307,1,娛樂場所_p02,娛樂場所_ape_1_42e5edbc,category_resampled,NaN,0.00,0.353307
6,學校,設計一組針對失語症患者的簡單、具體且容易聯想到答案的提示句列表。,0.635580,3,學校_ape_2_4f193443,學校_ape_3_e7a8d43d,category_resampled,NaN,0.00,0.635580
7,家具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.830629,0,NaN,家具_p02,category_meta,v2_discriminative,0.05,0.880629
8,廚房用品,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.609120,0,NaN,廚房用品_p02,category_meta,v2_discriminative,0.05,0.659120
9,文具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.739974,0,NaN,文具_p02,category_meta,v2_discriminative,0.05,0.789974


Categories with best prompts: 21
['交通工具', '休閒娛樂', '動作', '動物', '喝', '娛樂場所', '學校', '家具', '廚房用品', '文具', '服飾&配件', '水果', '生活用品', '調味', '輔具', '運動場所', '醫療場所', '金融機構', '電器用品', '食物', '餐具']


In [5]:
final_rows = []

for _, ex in tqdm(best_human_df.iterrows(), total=len(best_human_df), desc="Final generation by category"):
    subcategory = ex["subcategory"]

    if subcategory not in best_prompt_map:
        continue

    best_prompt_text = best_prompt_map[subcategory]

    demo_examples = sample_demo_examples(
        seed_df=seed_df,
        subcategory=subcategory,
        n_examples=3,
        random_state=42,
    )
    demo_block = format_demo_block(demo_examples)

    final_generated_cue = generate_therapy_cue(
        instruction=best_prompt_text,
        target_word=ex["target_word"],
        pos=ex["pos"],
        subcategory=subcategory,
        demo_block=demo_block,
    )

    score_dict = compute_proxy_score(
        generated_cue=final_generated_cue,
        target_word=ex["target_word"],
        reference_cue=ex["cue_text"],
    )

    final_rows.append({
        "target_word": ex["target_word"],
        "pos": ex["pos"],
        "main_category": ex["main_category"],
        "subcategory": subcategory,
        "reference_best_human_cue": ex["cue_text"],
        "best_prompt": best_prompt_text,
        "final_generated_cue": final_generated_cue,
        **score_dict,
    })

final_cues_df = pd.DataFrame(final_rows)

print("FINAL CUES SHAPE:", final_cues_df.shape)
display(final_cues_df.head(20))

Final generation by category: 100%|██████████| 130/130 [03:07<00:00,  1.44s/it]

FINAL CUES SHAPE: (125, 11)


,target_word,pos,main_category,subcategory,reference_best_human_cue,best_prompt,final_generated_cue,semantic_similarity,brevity_score,target_leak_penalty,proxy_score
0,上廁所,動詞,日常行為,動作,會坐/蹲在馬桶上執行動作,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,在離馬桶稍遠的地方準備好紙巾和毛巾，然後輕,0.536766,0.952381,0.0,0.661450
1,下象棋,動詞,日常活動,休閒娛樂,需要理解不同棋子的移動規則才能下得好,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,盤上落子，棋局起始，靜候勝負爭奪,0.531832,1.000000,0.0,0.672283
2,假牙,名詞,日常起居,輔具,剛戴上時可能會有異物感,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,,0.000000,0.000000,1.0,0.000000
3,傘,名詞,日常起居,生活用品,"用完後要晾乾,不然會發霉",你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標生活用品詞彙，並...,"需要防風防水,方便攜帶出行",0.288061,1.000000,0.0,0.501642
4,公園,名詞,地點,娛樂場所,適合散步、慢跑的區域,描述地點的常見景象或功能，避免太抽象的描述。,遊樂區有秋千和沙坑，小朋友們愛玩其中,0.467798,1.000000,0.0,0.627459
5,公車,名詞,日常起居,交通工具,票價便宜,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,上車後，找到自己的座位，系好安全帶,0.283050,1.000000,0.0,0.498135
6,冰淇淋,名詞,日常飲食,食物,吃起來冰涼黏稠,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標食物詞彙，並生成...,,0.000000,0.000000,1.0,0.000000
7,冰箱,名詞,日常起居,電器用品,"長方形,有兩個門",你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標電器用品詞彙，並...,冷風、冷藏、旋轉、放置桌上,0.395688,1.000000,0.0,0.576982
8,冷氣,名詞,日常起居,電器用品,開27度可以環保愛地球,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標電器用品詞彙，並...,,0.000000,0.000000,1.0,0.000000
9,刷牙,動詞,日常行為,動作,正確的XX方式是由上往下、由內到外,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,,0.000000,0.000000,1.0,0.000000


In [6]:
final_cues_df.to_csv(
    FINAL_GENERATED_CUES_BY_CATEGORY_PATH,
    index=False,
    encoding="utf-8-sig"
)
print("Saved:", FINAL_GENERATED_CUES_BY_CATEGORY_PATH)

Saved: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\final\final_generated_cues_by_category.csv


In [7]:
for subcat in sorted(final_cues_df["subcategory"].unique()):
    print("=" * 100)
    print("CATEGORY:", subcat)

    best_prompt_text = final_cues_df[final_cues_df["subcategory"] == subcat]["best_prompt"].iloc[0]
    print("\nBEST PROMPT:")
    print(best_prompt_text)

    print("\nTOP GENERATED CUES:")
    display(
        final_cues_df[final_cues_df["subcategory"] == subcat][
            ["target_word", "reference_best_human_cue", "final_generated_cue", "proxy_score"]
        ].sort_values("proxy_score", ascending=False).head(10)
    )

CATEGORY: 交通工具

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並生成具體、生活化、容易聯想到答案的提示句。

我的最終目標是幫助高齡失語症患者，透過搭乘經驗與用途線索，成功說出目標詞彙。

原則：
1. 優先描述用途、搭乘方式、常見情境。
2. 避免只描述太廣泛的移動概念。
3. 優先使用高齡者熟悉的搭車經驗與日常說法。
4. 避免太書面、太技術化的描述。

提示特徵資料集（交通工具專用）：
●用途：載人、代步、通勤、旅行、上學
●情境：在路上、停靠站牌、上下班、城市裡、出遠門
●辨識特徵：固定路線、很多人一起搭、司機開、有座位、有輪子
●熟悉場景：等車、刷卡、買票、坐著移動

生成限制：
●提示句中不得出現目標詞彙。
●每句長度限於 20 個中文字以內。
●需具體、自然、容易理解。
●盡量指向單一交通工具。

輸出格式：
●每次僅生成一個提示句。
●只輸出提示句本身，不要解釋。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
55,火車,"需要購票搭乘,有分自由座和對號座",需要購票才能登車,0.881714
46,汽車,需要加油才能行駛,需要車道才能行駛,0.705243
81,自行車,運動或代步皆適合,騎行方便，速度快，適合短途出行,0.684109
121,高鐵,"透過電力驅動,行駛於專用軌道上",需要刷卡才能進站,0.511423
5,公車,票價便宜,上車後，找到自己的座位，系好安全帶,0.498135
35,摩托車,需要加汽油或電力才能運行,騎行在城市街道上，感受風吹拂臉頰,0.421682


CATEGORY: 休閒娛樂

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一個簡短、具體、生活化且容易聯想到答案的提示句，不可直接說出目標詞。每次只輸出一個提示句，不要解釋。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
47,泡茶,選擇合適的茶具與水質,將水倒入茶壺，加入紅茶包和糖，輕輕,0.863611
1,下象棋,需要理解不同棋子的移動規則才能下得好,盤上落子，棋局起始，靜候勝負爭奪,0.672283
102,跳舞,與拍手、踩腳或轉圈的動作結合,在空曠場地隨意扭轉身體，與同伴互動，,0.655932
21,唱歌,"有時會搭配樂器,如吉他、鋼琴或卡拉OK伴奏",在空蕩的房間裡唱著歌，讓心情更加愉快,0.598839
101,購物,節日會推出促銷活動吸引客人,在超市裡逛逛，看看你想買什麼,0.572226
30,打拳,XX能提升身體平衡感,在公園裡揮舞手臂，感受風和陽光,0.457230
66,登山,"有可能會碰到熊,要有防熊噴霧",在山頂遠眺，感受風景無比壯闊,0.364310
41,散步,"天氣涼爽時會更舒服,避免烈日或下雨天",,0.000000
68,看電視,用遙控器來切換頻道或調整音量,,0.000000
67,看書,會用手翻頁或摺角作為書籤,,0.000000


CATEGORY: 動作

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成具體、容易理解、方便聯想到答案的提示句。

目標是讓高齡失語症患者透過動作情境、目的或流程聯想到該動作名稱。

原則：
1. 提示句應描述做這個動作時的典型情境。
2. 可描述目的、使用工具、動作前後會發生的事情。
3. 避免直接命令患者去做該動作。
4. 避免只說太抽象的動作概念。

提示特徵資料集（動作專用）：
●情境：早上起床後、吃飯前、洗澡後、出門前
●目的：讓身體乾淨、整理自己、完成日常生活
●工具：牙刷、剪刀、指甲剪、馬桶、毛巾
●流程：先做什麼，再做什麼
●結果：變乾淨、變整齊、完成整理

生成限制：
●提示句中不得出現目標詞彙。
●每句長度限於 20 個中文字以內。
●需具體、清楚、容易理解。
●應盡量指向單一動作。

輸出格式：
●每次僅生成一個提示句。
●只輸出提示句本身，不要解釋。

補充要求：請優先使用高齡者熟悉的生活經驗、家庭情境或日常場景來設計提示句。

TOP GENERATED CUES:


...

,target_word,reference_best_human_cue,final_generated_cue,proxy_score
59,熱水瓶,煮沸後會跳起來,蓋上蓋子，倒滿冷水，開始煮沸,0.906737
110,除濕機,會吹出熱熱的風,冷風直吹，除濕效果佳,0.842393
117,電鍋,內鍋、外鍋、加熱板、保溫開關,煮飯、加熱食物、炊事工具,0.840463
65,瓦斯爐,鍋子、鍋鏟,火鍋、調味料、煮飯,0.736919
118,電風扇,插電後即可使用,吹風、清潔、擺放桌面,0.724218
116,電話,有回撥功能,手柄、連接線、插座,0.656177
34,插座,通電,插電、放在桌上、廚房常見,0.635623
7,冰箱,"長方形,有兩個門",冷風、冷藏、旋轉、放置桌上,0.576982
114,電燈,會連接開關,壁挂式、暖光、晚上使用,0.569337
19,吹風機,"部分有折疊設計,方便攜帶",夏天，厨房，挂牆，吹乾頭髮,0.528681


CATEGORY: 食物

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標食物詞彙，並生成具體、容易理解、貼近生活的提示句，幫助患者成功說出答案。

目標是讓高齡失語症患者透過熟悉的飲食經驗聯想到該食物名稱。

原則：
1. 提示句需具體、生活化、貼近日常飲食經驗。
2. 避免抽象或過於廣泛的描述。
3. 每句應盡量指向該食物的專屬特徵。

提示特徵資料集（食物專用）：
●味道與口感：甜、鹹、酥、軟、冰、熱
●外觀特徵：片狀、圓形、白色、金黃色
●用餐情境：早餐、點心、肚子不舒服時吃
●料理與食用方式：烤、煎、塗果醬、配牛奶
●熟悉人物或生活場景：早餐店、家裡餐桌、媽媽準備

生成限制：
●提示句中不得出現目標詞彙。
●每句長度限於 20 個中文字以內。
●需具體且容易理解。
●避免太廣泛，盡量指向單一食物。

輸出格式：
●每次僅生成一個提示句。
●只輸出提示句本身，不要解釋。

補充要求：請優先使用高齡者熟悉的生活經驗、家庭情境或日常場景來設計提示句。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
124,麵,由小麥製成的主食,一種由小麥製成的主食,0.991003
54,漢堡,西式餐點,一種肉質鮮嫩多汁的食物,0.819035
53,湯圓,一種由糯米粉製成的食品,一種甜而不咸的小圓形食品,0.818176
45,水餃,"麵皮包著內餡,冷凍儲存",一種皮薄馅大的食物,0.788069
75,米,煮後變軟可食用,一種由稻穀磨碎後加水蒸煮而成的主食,0.756786
98,豆花,一種用黃豆製成的甜點,一種由大豆製成的冷飲，通常搭配冰糖和薄荷,0.741191
97,豆腐,可煎、炸、燉、炒,一種由黄豆製成的塊狀食品,0.729688
112,雞排,吃起來香脆多汁,一種外觀呈片狀、金黃色的食品,0.690983
62,牛肉麵,台灣特色小吃,一種紅色、長條形、略帶彈性的主食,0.684537
17,吐司,拉肚子的時候會吃的,一種薄脆的面包片，通常搭配奶油或果酱食用,0.661863


CATEGORY: 餐具

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一個簡短、具體、生活化且容易聯想到答案的提示句，不可直接說出目標詞。每次只輸出一個提示句，不要解釋。

補充要求：請優先選擇最有辨識度、最能區分其他同類詞彙的特徵，避免過於廣泛的描述。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
52,湯匙,"嬰兒用的特別小,邊緣圓滑,方便餵食",使用時輕拿輕放，保持平穩,0.567202
15,叉子,兒童款的XX會比較圓潤,,0.000000
72,碗,可以是陶瓷、不鏽鋼、塑膠或木頭材質製成的,,0.000000


In [8]:
output_txt = PROJECT_ROOT / "data" / "final" / "best_prompts_and_cues_by_category.txt"

with open(output_txt, "w", encoding="utf-8") as f:
    for subcat in sorted(final_cues_df["subcategory"].unique()):
        f.write("=" * 100 + "\n")
        f.write(f"CATEGORY: {subcat}\n\n")

        best_prompt_text = final_cues_df[final_cues_df["subcategory"] == subcat]["best_prompt"].iloc[0]
        f.write("BEST PROMPT:\n")
        f.write(str(best_prompt_text) + "\n\n")

        f.write("TOP GENERATED CUES:\n")
        top_rows = final_cues_df[final_cues_df["subcategory"] == subcat].sort_values(
            "proxy_score", ascending=False
        ).head(10)

        for _, r in top_rows.iterrows():
            f.write(f"- target_word: {r['target_word']}\n")
            f.write(f"  reference_best_human_cue: {r['reference_best_human_cue']}\n")
            f.write(f"  final_generated_cue: {r['final_generated_cue']}\n")
            f.write(f"  proxy_score: {r['proxy_score']}\n\n")

print("Saved:", output_txt)

Saved: c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\final\best_prompts_and_cues_by_category.txt
